In [1]:
import pandas as pd
import json

In [2]:
import warnings
warnings.filterwarnings('ignore')

In [91]:
# Load the Excel file
beverage_list_path = './TCF-Coffe_App-Export_Beverage-List.csv'
impact_data_path = './TCF-Coffe_App-Export_Impact-Data.csv'
sugar_data_path = './TCF-Coffe_App-Export_Sugar-Data.csv'
impact_description_path = './TCF-Coffe_App-Export_Impact-Description.csv'
coffee_description_path = './TCF-Coffe_App-Export_Coffee-Description.csv'

df_beverage_list = pd.read_csv(beverage_list_path,delimiter=';',encoding='windows-1252')
df_impact_data = pd.read_csv(impact_data_path,delimiter=';',encoding='windows-1252')
df_sugar_data = pd.read_csv(sugar_data_path,delimiter=';',encoding='windows-1252')
df_impact_description = pd.read_csv(impact_description_path,delimiter=';',encoding='windows-1252')
df_coffee_description = pd.read_csv(coffee_description_path,delimiter=';',encoding='windows-1252')

df_beverage_list['Labels'] = df_beverage_list['Labels'].str.split(';').str[0]
df_beverage_list['Labels'] = df_beverage_list['Labels'].str.replace(' ', '', regex=False).str.replace(',', '|', regex=False).str.replace('.', '', regex=False)
df_beverage_list['Retail name'] = df_beverage_list['Retail name'] + "\n" + df_beverage_list['Salepoint']

df_beverage_list.head()

,Beverage ID,Retail name,Retail price,Hidden Costs,True Price,Beverage-type,Salepoint,Deca,Milk Type,Price Excluding Tax,Value Added Tax,Smart Value Added Tax,Smart Pricing rounded,Labels
0,"Café Blue Planet, Dallmayr",Café Blue Planet\nDallmayr,1.7,-0.014,1.69,Café,Dallmayr,False,none,1.573,0.127,0.025,1.60,blue-planet|fairtrade|eu-organic
1,"Café Macchiato Blue Planet, Dallmayr",Café Macchiato Blue Planet\nDallmayr,1.8,0.041,1.84,Café Macchiato,Dallmayr,False,none,1.665,0.135,0.030,1.70,blue-planet|fairtrade|eu-organic
2,"Café Macchiato Via Verde, Dallmayr",Café Macchiato Via Verde\nDallmayr,1.7,0.148,1.85,Café Macchiato,Dallmayr,False,none,1.573,0.127,0.040,1.65,via-verde|fairtrade|eu-organic
3,"Café Via Verde, Dallmayr",Café Via Verde\nDallmayr,1.6,0.094,1.69,Café,Dallmayr,False,none,1.480,0.120,0.035,1.55,via-verde|fairtrade|eu-organic
4,"Café, Compass Machine",Café Via Verde\nCompass Machine,1.5,0.094,1.59,Café,Compass Machine,False,none,1.388,0.112,0.035,1.45,via-verde|fairtrade|eu-organic


In [11]:

duplicate_rows = df_impact_data.groupby(by=["Beverage ID", "Ingredient", "Impact Category", "Stage", "Indicator"]).filter(lambda x: len(x) >= 2)
duplicate_rows = duplicate_rows.sort_values(by=["Beverage ID", "Ingredient", "Impact Category", "Stage", "Indicator"])
duplicate_rows.to_csv('duplicate_rows.csv',index=False)

In [10]:
duplicate_rows

,Beverage ID,Ingredient,Stage,Impact Category,Indicator,Unit,Monetary Value,Value,References
26295,"Café Blue Planet, Dallmayr",Coffee beans,Offsetting schemes,Biodiversity,Biodiversity restoration through Reforestation...,m²/kg,0.0,0.0,"Fairtrade regulation, No specific regulation e..."
26297,"Café Blue Planet, Dallmayr",Coffee beans,Offsetting schemes,Biodiversity,Biodiversity restoration through Reforestation...,m²/kg,0.0,0.0,"EU-organic standards, No specific regulation o..."
26457,"Café Blue Planet, Dallmayr",Coffee beans,Offsetting schemes,Biodiversity,Biodiversity restoration through Reforestation...,m²/kg,0.0,0.0,"Fairtrade regulation, No specific regulation e..."
26459,"Café Blue Planet, Dallmayr",Coffee beans,Offsetting schemes,Biodiversity,Biodiversity restoration through Reforestation...,m²/kg,0.0,0.0,"EU-organic standards, No specific regulation o..."
26655,"Café Blue Planet, Dallmayr",Coffee beans,Offsetting schemes,Biodiversity,Biodiversity restoration through Reforestation...,m²/kg,0.0,0.0,"Fairtrade regulation, No specific regulation e..."
...,...,...,...,...,...,...,...,...,...
33399,"Ristretto, Le Klee",Coffee beans,Consumption,Health,Thiamethoxam,mg/kg,0.0,0.0,Sustainability Impact Metrics (a spin-off of D...
33596,"Ristretto, Le Klee",Coffee beans,Consumption,Health,Thiamethoxam,mg/kg,0.0,0.0,Sustainability Impact Metrics (a spin-off of D...
33597,"Ristretto, Le Klee",Coffee beans,Consumption,Health,Thiamethoxam,mg/kg,0.0,0.0,Sustainability Impact Metrics (a spin-off of D...
33758,"Ristretto, Le Klee",Coffee beans,Consumption,Health,Thiamethoxam,mg/kg,0.0,0.0,Sustainability Impact Metrics (a spin-off of D...


In [92]:
def get_indicator_values(row):

    # # Find matches where df_impact_description['Indicator'] is contained in row['Indicator']
    # matched_rows = df_impact_description[
    #     df_impact_description['Indicator'].apply(lambda x: x in row['Indicator'])
    # ]
    
    # if matched_rows.empty:
    #     impact_definition = None
    #     monetisation_method = None
    # else:
    #     # Select the most specific match (longest string in df_impact_description['Indicator'])
    #     impact_definition = matched_rows.loc[matched_rows['Indicator'].str.len().idxmax(), 'Indicator definition']
    #     monetisation_method = matched_rows.loc[matched_rows['Indicator'].str.len().idxmax(), 'Monetisation method']
    
    return {
        'indicators': row['Indicator'],
        'unit': row['Unit'],
        'impactValue': row['Value'], 
        'costValue': row['Monetary Value'], 
        # 'impactDefinition': impact_definition,
        # 'monetisationMethod': monetisation_method,
        'reference': None if pd.isna(row['References']) else row['References']
    }


def calculate_impacts(group):
    # Group by stage and impact-category
    grouped_impacts = group.groupby(['Ingredient','Stage', 'Impact Category']).apply(lambda x: {
        'stage': x.iloc[0]['Stage'],
        'ingredient': x.iloc[0]['Ingredient'],
        'impactCategory': x.iloc[0]['Impact Category'],
        'impactValue': x['Value'].sum(),   # Sum impact values
        'costValue': x['Monetary Value'].sum(),   # Sum cost values
        'details': x.apply(get_indicator_values, axis=1).tolist()
    }).reset_index(drop=True).tolist()

    return grouped_impacts

In [83]:
df_beverage_list.head()

,Beverage ID,Retail name,Retail price,Hidden Costs,True Price,Beverage-type,Salepoint,Deca,Milk Type,Price Excluding Tax,Value Added Tax,Smart Value Added Tax,Smart Pricing rounded,Labels
0,"Café Blue Planet, Dallmayr",Café Blue PlanetDallmayr,1.7,-0.014,1.69,Café,Dallmayr,False,none,1.573,0.127,0.025,1.60,blue-planet|fairtrade|eu-organic
1,"Café Macchiato Blue Planet, Dallmayr",Café Macchiato Blue PlanetDallmayr,1.8,0.041,1.84,Café Macchiato,Dallmayr,False,none,1.665,0.135,0.030,1.70,blue-planet|fairtrade|eu-organic
2,"Café Macchiato Via Verde, Dallmayr",Café Macchiato Via VerdeDallmayr,1.7,0.148,1.85,Café Macchiato,Dallmayr,False,none,1.573,0.127,0.040,1.65,via-verde|fairtrade|eu-organic
3,"Café Via Verde, Dallmayr",Café Via VerdeDallmayr,1.6,0.094,1.69,Café,Dallmayr,False,none,1.480,0.120,0.035,1.55,via-verde|fairtrade|eu-organic
4,"Café, Compass Machine",Café Via VerdeCompass Machine,1.5,0.094,1.59,Café,Compass Machine,False,none,1.388,0.112,0.035,1.45,via-verde|fairtrade|eu-organic


In [93]:
# Transform the data
import os
import re


formatted_data = []

i = 0;

# milk_beverage_lists = ['Cappuccino',"Chocolait","Latte Macchiato","Macchiato","Mocaccino", 'Renversé']

for _, beverage in df_beverage_list.iterrows():
    beverage_id = beverage['Beverage ID']
    retail_name = beverage['Retail name']

    # all_removals = labels_list + ['décaféiné', "lait d'amande", "lait d'avoine", "lait de soja", "sans-lactose"]

    # recipe_id_pattern = r'(' + '|'.join(map(re.escape, all_removals)) + r')'
    # recipe_id = re.sub(recipe_id_pattern, '', retail_name, flags=re.IGNORECASE).replace(',','').strip()
    recipe_id = beverage['Beverage-type']
    # sale_point_id = sale_point_mapping[beverage['Salepoint']]
    # is_decaf = 'décaféiné' in beverage['Retail name'].lower()
    is_decaf = beverage['Deca']
    # has_milk = any(x in beverage['Retail name'] for x in milk_beverage_lists)

    # Determine the type of milk, if present
    # milk_types = {'lait d\'amande': 'Almond', 'lait d\'avoine': 'Oat', 'lait de soja': 'Soy', 'sans-lactose': 'Lactose-Free'}
    # milk_type = None
    # for key, value in milk_types.items():
    #     if key in beverage['Retail name'].lower():
    #         milk_type = value
    #         break
    # milk_type = milk_type if milk_type else 'Dairy' if any(x in beverage['Retail name'].lower() for x in ['milk', 'latte', 'cappuccino', 'renversé']) else None
        
    milk_type = beverage['Milk Type']
    has_milk = milk_type != 'none'

    definition_row = df_coffee_description[df_coffee_description['Recipe'] == recipe_id]
    definition = definition_row['Definition'].iloc[0] if not definition_row.empty else ""


    impact_rows = df_impact_data[df_impact_data['Beverage ID'] == beverage_id]
    # print(impact_rows)
    ingredient_list = impact_rows['Ingredient'].unique().tolist()



    grouped_impacts = calculate_impacts(impact_rows)

    path_impacts = './results/impacts/'+beverage_id.lower().replace(' ','_').replace(',','')+'.json'
    os.makedirs(os.path.dirname(path_impacts), exist_ok=True)
    with open(path_impacts, 'w', encoding='utf-8') as f:
        json.dump(grouped_impacts, f, indent=4, ensure_ascii=False)



    formatted_data.append({
        'serveId': beverage_id,
        'recipeId': recipe_id,
        'retailName': retail_name,
        'retailPrice': beverage['Retail price'],
        'hiddenCost': beverage['Hidden Costs'],
        'truePrice': beverage['True Price'],
        'labels': beverage['Labels'],
        'isDecaf': is_decaf,
        'hasMilk': has_milk,
        'milkType': milk_type,
        'coffeeDetails': definition, 

    })

# Create the formatted DataFrame
formatted_df = pd.DataFrame(formatted_data)

# Save to CSV
output_path = './results/coffee_data.csv'
formatted_df.to_csv(output_path, index=False)



print(f"Formatted data saved to {output_path}")

Formatted data saved to ./results/coffee_data.csv


In [94]:

list_sugars = ["Swiss sugar default","Swiss sugar low","Swiss sugar moderate","Swiss sugar high"]

for sugar in list_sugars:
    sugar_rows = df_sugar_data[df_sugar_data['Beverage ID'] == sugar]
    print(sugar_rows)

    grouped_impacts = calculate_impacts(sugar_rows)

    path_impacts = './results/sugar/'+sugar.lower().replace(' ','_').replace(',','')+'.json'
    os.makedirs(os.path.dirname(path_impacts), exist_ok=True)
    with open(path_impacts, 'w', encoding='utf-8') as f:
        json.dump(grouped_impacts, f, indent=4, ensure_ascii=False)


             Beverage ID      Country Ingredient Labels        Stage  \
0    Swiss sugar default  Switzerland  Sugarbeet   none   Production   
1    Swiss sugar default  Switzerland  Sugarbeet   none   Production   
2    Swiss sugar default  Switzerland  Sugarbeet   none   Production   
3    Swiss sugar default  Switzerland  Sugarbeet   none   Production   
4    Swiss sugar default  Switzerland  Sugarbeet   none   Production   
..                   ...          ...        ...    ...          ...   
195  Swiss sugar default  Switzerland  Sugarbeet   none  Consumption   
196  Swiss sugar default  Switzerland  Sugarbeet   none  Consumption   
197  Swiss sugar default  Switzerland  Sugarbeet   none  Consumption   
198  Swiss sugar default  Switzerland  Sugarbeet   none  Consumption   
199  Swiss sugar default  Switzerland  Sugarbeet   none  Consumption   

    Impact Category                          Indicator         Unit  Value  \
0       Environment  Fine particulate matter formation  k

In [95]:
output_path= "./results/impacts_definitions.csv"
df_impact_description.to_csv(output_path, index=False)

In [96]:
import shutil

source_dir = './results/'
destination_dir = '../public/data/'

# Copy the contents of the source directory to the destination directory
shutil.copytree(source_dir, destination_dir, dirs_exist_ok=True)

print(f"Contents of {source_dir} copied to {destination_dir}")

Contents of ./results/ copied to ../public/data/
